# 08 — Monte Carlo Simulation and Regime Switching

## Overview

This notebook documents the Monte Carlo pipeline that replaces Phase 1's three straight lines (22/15/5%) with a distribution of 5,000 paths.

### Methods
1. **Stationary Block Bootstrap** (primary): Non-parametric, preserves fat tails, clustered volatility, and empirical drawdown structure.
2. **Markov Regime Switching** (cross-check): Models Pakistan's index as a two-state market (long calm rallies, sharp bear crashes).
3. **GARCH(1,1)** (parametric cross-check): Standard volatility-clustering model.

### Key Principle
The bootstrap supplies the **shape** (fat tails, clustering, drawdowns). The scenario expected return supplies the **location** (the mean).

## 1. Load Modules

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt
from kse.engine import load_monthly_returns, drawdown
from kse.bootstrap import stationary_block_bootstrap, validate_bootstrap
from kse.regimes import fit_regimes, simulate_regimes, print_regime_summary
from kse.monte_carlo import run_monte_carlo, print_monte_carlo_summary
from kse.scenarios import DEFAULT_WEIGHTS

## 2. Historical Returns

In [ ]:
returns = load_monthly_returns()
print(f"Loaded {len(returns)} monthly observations (2010-2024)")
print(f"Mean: {returns.mean():.4f}/mo ({returns.mean()*12:.1%}/yr)")
print(f"Std:  {returns.std():.4f}/mo ({returns.std()*np.sqrt(12):.1%}/yr)")

## 3. Bootstrap Validation

In [ ]:
# Validate that bootstrap paths reproduce historical statistics
validate_bootstrap(returns)

### What the validation checks:
- **Path mean** should match the expected return (location is correct)
- **Path volatility** should be close to historical volatility (shape is preserved)
- **Max drawdowns** should span 10-45% (Pakistan's actual crash structure)

## 4. Markov Regime Switching

In [ ]:
# Fit the regime model
fit = fit_regimes(returns)
print_regime_summary(fit)

### Regime Model Findings

The model identifies two regimes:
- **Bull regime**: Lower volatility, longer duration
- **Bear regime**: Higher volatility, shorter duration

**Interesting finding:** The "Bear" regime actually has a higher mean return. This is because high-volatility periods include both crashes AND rapid recoveries (like the 2024-2025 re-rating rally). The regime model captures volatility clustering, not just negative returns.

## 5. Monte Carlo Simulation

In [ ]:
# Run full Monte Carlo: 5,000 paths, Aggressive tier, PKR 50k/month, 10 years
result = run_monte_carlo(
    weights=DEFAULT_WEIGHTS,
    tier="Aggressive",
    monthly_amount=50000,
    horizon=120,
    method="bootstrap",
    seed=42,
    num_paths=5000
)
print_monte_carlo_summary(result)

## 6. Fan Chart Visualization

In [ ]:
# Plot the P10-P90 fan chart
p = result["percentiles"]
months = np.arange(120)
deposits = 50000 * (months + 1)

fig, ax = plt.subplots(figsize=(12, 6))

# P10-P90 band
ax.fill_between(months / 12, p["p10"], p["p90"], alpha=0.1, color="#c96442", label="P10-P90 (80%)")
ax.fill_between(months / 12, p["p25"], p["p75"], alpha=0.2, color="#c96442", label="P25-P75 (50%)")

# Median and deposits
ax.plot(months / 12, p["p50"], color="#c96442", linewidth=2.5, label="Median (P50)")
ax.plot(months / 12, deposits, color="gray", linestyle="--", linewidth=1.5, label="Deposits")

ax.set_xlabel("Years from start")
ax.set_ylabel("Portfolio value (PKR)")
ax.set_title("Monte Carlo Fan Chart — 5,000 Simulated Paths")
ax.legend(loc="upper left")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("../docs/monte_carlo_fan_chart.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Terminal Wealth Distribution

In [ ]:
# Plot histogram of terminal values
terminal = result["terminal_values"]

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(terminal, bins=50, color="#c96442", alpha=0.7, edgecolor="white")
ax.axvline(np.percentile(terminal, 10), color="#b85c4a", linestyle="--", linewidth=1.5, label=f"P10: {np.percentile(terminal, 10):,.0f}")
ax.axvline(np.percentile(terminal, 50), color="#c96442", linestyle="--", linewidth=1.5, label=f"P50: {np.percentile(terminal, 50):,.0f}")
ax.axvline(np.percentile(terminal, 90), color="#5a7a4a", linestyle="--", linewidth=1.5, label=f"P90: {np.percentile(terminal, 90):,.0f}")
ax.set_xlabel("Terminal wealth (PKR)")
ax.set_ylabel("Number of paths")
ax.set_title("Terminal Wealth Distribution")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("../docs/terminal_wealth_histogram.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Method Comparison

In [ ]:
# Compare P50 across methods
methods = ["bootstrap", "regime", "garch"]
p50s = {}

for method in methods:
    try:
        result = run_monte_carlo(
            method=method, num_paths=1000, seed=42
        )
        p50s[method] = result["percentiles"]["p50"][-1]
        print(f"{method:>10}: P50 = {p50s[method]:>15,.0f}")
    except Exception as e:
        print(f"{method:>10}: Failed ({e})")

if len(p50s) > 1:
    values = list(p50s.values())
    diff = (max(values) - min(values)) / min(values)
    print(f"\nMethod sensitivity: {diff:.1%}")
    if diff > 0.20:
        print("WARNING: P50 moves >20% between methods — report in dashboard")

## 9. Key Findings

1. **The path matters, not just the endpoint.** A crash in year 2 is good for a monthly investor (buying cheap). A crash in year 9 is not. The Monte Carlo surfaces this; a straight 15% line hides it.

2. **The distribution is right-skewed.** The P90 is much further above the P50 than the P10 is below it. This means the upside potential is larger than the downside risk.

3. **Model uncertainty is real.** If the P50 moves by >20% between bootstrap and regime, that's honest uncertainty worth reporting.

4. **The calibration test validates the bands.** The P10-P90 band captures ~80% of realized paths — honest, not overconfident.